# Feature engineering

Our Jupyter notebook, Error_Analysis ran a error and fairness analysis on our Neural Network and LightGBM model. Creating a threshold we discovered that both models comparitivly performed the same with LightGBM having 50 false alarms. The most profound discovery in our error analysis was that our models were struggling to capture signal for sudden applicant behavioral changes, for example, some applicants may have been good at paying back loans or had a good credit history but suddenly found themselves struggling to pay back debt. 

In [1]:
import sqlite3
import time 
from pathlib import Path

import numpy as np
import pandas as pd

DB_PATH = Path(r"C:\Data_Projects\Project_2_Data4401\home_credit.db")
PROCESSED = Path(r"C:\Data_Projects\Project_2_Data4401\data\processed")

conn = sqlite3.connect(DB_PATH)
# Setting up 
conn.execute("PRAGMA mmap_size = 4000000000")
conn.execute("PRAGMA cache_size = -2000000")
conn.execute("PRAGMA temp_store = MEMORY")

# Current Application

In [2]:
def run_query(name, sql):
    start = time.time()
    df = pd.read_sql(sql, conn)
    print(f"{name}: {len(df):,} rows x {df.shape[1]} columns in {time.time() - start:.0f}s")
    return df

In [3]:
BASE_DATA = """
SELECT *
FROM v_application_full
""" # everything merged with each other already
df_base = run_query("base", BASE_DATA)

base: 356,255 rows x 123 columns in 46s


In [4]:
df_base["NO_EMPLOYER"] = (df_base["DAYS_EMPLOYED"] == 365243).astype(int) #365243 is a placeholder also makes binary
df_base["DAYS_EMPLOYED"] = (df_base["DAYS_EMPLOYED"].replace(365243, np.nan))

print(df_base["dataset"].value_counts())
print(f"Clients with no TARGET (should equal the test count): {df_base['TARGET'].isna().sum():,}")


dataset
train    307511
test      48744
Name: count, dtype: int64
Clients with no TARGET (should equal the test count): 48,744


# Applicant History
### Work Being Done:
1. The client's past: other lenders and earlier Home Credit Loans
2. The 5 history tables (burea & balance, previous_application, pos_cash_balance, credit_card_balance, installments_payments)
3. **Aggregated** to one client 

| Source table | Rows before | Summary features |
|---|---:|---:|
| `bureau` + `bureau_balance` | 1.7M + 27.3M | 15 |
| `previous_application` | 1.7M | 11 |
| `pos_cash_balance` | 10.0M | 5 |
| `credit_card_balance` | 3.8M | 9 |
| `installments_payments` | 13.6M | 7 |
| **Result** | **~57M** | **356,255 applicants × 46 features** |


In [5]:
HISTORY_SQL = """
WITH bb AS (
    SELECT SK_ID_BUREAU,
        COUNT(*) AS bb_months,
        AVG(STATUS IN ('1','2','3','4','5')) AS bb_pct_late,
        MAX(CASE WHEN STATUS IN ('1','2','3','4','5') THEN CAST(STATUS AS INT) ELSE 0 END) AS bb_worst_status
    FROM bureau_balance
    GROUP BY SK_ID_BUREAU
),

bureau_agg AS (
    SELECT b.SK_ID_CURR,
        COUNT(*) AS bureau_n_loans,
        SUM(b.CREDIT_ACTIVE = 'Active') AS bureau_n_active,
        SUM(b.CREDIT_ACTIVE = 'Closed') AS bureau_n_closed,
        SUM(b.AMT_CREDIT_SUM) AS bureau_total_credit,
        SUM(b.AMT_CREDIT_SUM_DEBT) AS bureau_total_debt,
        SUM(b.AMT_CREDIT_SUM_OVERDUE) AS bureau_total_overdue,
        MAX(b.AMT_CREDIT_MAX_OVERDUE) AS bureau_max_overdue,
        SUM(b.AMT_CREDIT_SUM_DEBT) / NULLIF(SUM(b.AMT_CREDIT_SUM), 0) AS bureau_debt_ratio,
        MIN(b.DAYS_CREDIT) AS bureau_oldest_loan_days,
        MAX(b.DAYS_CREDIT) AS bureau_newest_loan_days,
        MAX(b.CREDIT_DAY_OVERDUE) AS bureau_max_days_overdue,
        SUM(b.CNT_CREDIT_PROLONG) AS bureau_n_prolonged,
        AVG(bb.bb_pct_late) AS bureau_avg_pct_months_late,
        MAX(bb.bb_worst_status) AS bureau_worst_status
    FROM bureau AS b
    LEFT JOIN bb ON b.SK_ID_BUREAU = bb.SK_ID_BUREAU
    GROUP BY b.SK_ID_CURR
),

prev_agg AS (
    SELECT SK_ID_CURR,
        COUNT(*) AS prev_n_apps,
        SUM(NAME_CONTRACT_STATUS = 'Approved') AS prev_n_approved,
        SUM(NAME_CONTRACT_STATUS = 'Refused') AS prev_n_refused,
        AVG(NAME_CONTRACT_STATUS = 'Refused') AS prev_pct_refused,
        AVG(AMT_APPLICATION) AS prev_avg_amt_applied,
        AVG(AMT_CREDIT) AS prev_avg_amt_credit,
        AVG(AMT_CREDIT / NULLIF(AMT_APPLICATION, 0)) AS prev_avg_credit_to_applied,
        AVG(AMT_ANNUITY) AS prev_avg_annuity,
        AVG(CNT_PAYMENT) AS prev_avg_n_payments,
        MAX(DAYS_DECISION) AS prev_last_decision_days,
        AVG(RATE_DOWN_PAYMENT) AS prev_avg_down_payment_rate
    FROM previous_application
    GROUP BY SK_ID_CURR
),

pos_agg AS (
    SELECT SK_ID_CURR,
        COUNT(DISTINCT SK_ID_PREV) AS pos_n_loans,
        COUNT(*) AS pos_n_months,
        MAX(SK_DPD) AS pos_max_dpd,
        AVG(SK_DPD > 0) AS pos_pct_months_dpd,
        AVG(CNT_INSTALMENT_FUTURE) AS pos_avg_instalments_left
    FROM pos_cash_balance
    GROUP BY SK_ID_CURR
),

cc_agg AS (
    SELECT SK_ID_CURR,
        COUNT(DISTINCT SK_ID_PREV) AS cc_n_cards,
        COUNT(*) AS cc_n_months,
        AVG(AMT_BALANCE) AS cc_avg_balance,
        MAX(AMT_BALANCE) AS cc_max_balance,
        AVG(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0)) AS cc_avg_utilization,
        MAX(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0)) AS cc_max_utilization,
        AVG(AMT_DRAWINGS_CURRENT) AS cc_avg_drawings,
        MAX(SK_DPD) AS cc_max_dpd,
        AVG(SK_DPD > 0) AS cc_pct_months_dpd
    FROM credit_card_balance
    GROUP BY SK_ID_CURR
),

inst_agg AS (
    SELECT SK_ID_CURR,
        COUNT(*) AS inst_n_payments,
        AVG(DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT) AS inst_pct_late,
        AVG(MAX(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT, 0)) AS inst_avg_days_late,
        MAX(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT) AS inst_max_days_late,
        AVG(AMT_PAYMENT < AMT_INSTALMENT) AS inst_pct_underpaid,
        SUM(MAX(AMT_INSTALMENT - AMT_PAYMENT, 0)) AS inst_total_shortfall,
        AVG(AMT_PAYMENT / NULLIF(AMT_INSTALMENT, 0)) AS inst_avg_paid_ratio
    FROM installments_payments
    GROUP BY SK_ID_CURR
)

SELECT *
FROM (SELECT SK_ID_CURR FROM application) AS a
LEFT JOIN bureau_agg USING (SK_ID_CURR)
LEFT JOIN prev_agg USING (SK_ID_CURR)
LEFT JOIN pos_agg USING (SK_ID_CURR)
LEFT JOIN cc_agg USING (SK_ID_CURR)
LEFT JOIN inst_agg USING (SK_ID_CURR)
"""

history = run_query("history summaries", HISTORY_SQL)


history summaries: 356,255 rows x 47 columns in 274s


# Applicant Payament Change
**Purpose of this cell**: This will allow us to create signal for the model to pick up on when there is a change of payament behavior, and whether it is gettting better or worse. So the cell will look at a client who paid well for 4 years and if they happen to have a slip up all of a sudden sometime like spring time, it will still mark them down as a reliable applicant who pays on time. The cell will produce a signal if a long timeapplicant that had good standing history or bad suddenly changes for a long period up till now. This way it ensures that our model is able to capture the change within a applicants history to improve the model's ability to classify applicants who will and will not struggle paying on time. 

In [6]:
INST_CHANGE_SQL = """
WITH p AS (
    SELECT SK_ID_CURR,
        DAYS_INSTALMENT,
        (DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT) AS is_late,
        MAX(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT, 0) AS days_late,
        (AMT_PAYMENT < AMT_INSTALMENT) AS is_underpaid,
        AMT_PAYMENT
    FROM installments_payments
)
SELECT SK_ID_CURR,
    SUM(DAYS_INSTALMENT >= -180) AS inst_n_6m,
    SUM(DAYS_INSTALMENT >= -365) AS inst_n_12m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -180 THEN is_late END) AS inst_pct_late_6m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN is_late END) AS inst_pct_late_12m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -180 THEN days_late END) AS inst_avg_days_late_6m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN days_late END) AS inst_avg_days_late_12m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN is_underpaid END) AS inst_pct_underpaid_12m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -180 THEN is_late END) - AVG(is_late) AS inst_late_trend_6m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN is_late END) - AVG(is_late) AS inst_late_trend_12m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN days_late END) - AVG(days_late) AS inst_days_late_trend_12m,
    MAX(CASE WHEN is_late = 1 THEN DAYS_INSTALMENT END) AS inst_last_late_days,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN AMT_PAYMENT END)
        / NULLIF(AVG(CASE WHEN DAYS_INSTALMENT < -365 THEN AMT_PAYMENT END), 0) AS inst_payment_change_ratio
FROM p
GROUP BY SK_ID_CURR

"""
inst_change = run_query("installment change", INST_CHANGE_SQL)
inst_change.describe().T


installment change: 339,587 rows x 13 columns in 81s


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,339587.0,278154.892278,102880.492598,100001.000000,189042.500000,278238.000000,367315.500000,456255.000000
inst_n_6m,339587.0,4.890008,5.624901,0.000000,0.000000,4.000000,7.000000,122.000000
inst_n_12m,339587.0,10.148386,10.470531,0.000000,0.000000,8.000000,14.000000,176.000000
inst_pct_late_6m,211208.0,0.048401,0.116430,0.000000,0.000000,0.000000,0.000000,1.000000
inst_pct_late_12m,252727.0,0.052381,0.112542,0.000000,0.000000,0.000000,0.050000,1.000000
inst_avg_days_late_6m,211208.0,0.183872,0.997526,0.000000,0.000000,0.000000,0.000000,115.000000
inst_avg_days_late_12m,252727.0,0.236666,1.213447,0.000000,0.000000,0.000000,0.083333,159.461538
inst_pct_underpaid_12m,252727.0,0.059697,0.151166,0.000000,0.000000,0.000000,0.000000,1.000000
inst_late_trend_6m,211208.0,-0.017592,0.102812,-0.882353,-0.045455,0.000000,0.000000,0.982143
inst_late_trend_12m,252727.0,-0.015647,0.088983,-0.882353,-0.030925,0.000000,0.000000,0.960784


# Credit Card Changes Features


**PURPOSE of this cell** Financial strain building up on credit cards before the application. This sql command is looking into two things: whether an applicant is running at the limit and paying only the minimum.

In [8]:
CC_CHANGE_SQL = """
WITH c AS (
    SELECT SK_ID_CURR,
        MONTHS_BALANCE,
        AMT_BALANCE,
        AMT_DRAWINGS_CURRENT,
        SK_DPD,
        AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0) AS util,
        CASE WHEN AMT_INST_MIN_REGULARITY > 0
             THEN (AMT_PAYMENT_TOTAL_CURRENT <= AMT_INST_MIN_REGULARITY) END AS paid_min_only
    FROM credit_card_balance
)
SELECT SK_ID_CURR,
    SUM(MONTHS_BALANCE >= -12) AS cc_n_months_12m,
    AVG(CASE WHEN MONTHS_BALANCE >= -3 THEN util END) AS cc_util_3m,
    AVG(CASE WHEN MONTHS_BALANCE >= -12 THEN util END) AS cc_util_12m,
    AVG(CASE WHEN MONTHS_BALANCE >= -3 THEN util END) - AVG(util) AS cc_util_trend_3m,
    AVG(CASE WHEN MONTHS_BALANCE >= -12 THEN util >= 0.9 END) AS cc_pct_maxed_12m,
    AVG(CASE WHEN MONTHS_BALANCE >= -3 THEN AMT_BALANCE END)
        - AVG(CASE WHEN MONTHS_BALANCE BETWEEN -12 AND -4 THEN AMT_BALANCE END) AS cc_balance_change_3m,
    AVG(CASE WHEN MONTHS_BALANCE >= -3 THEN AMT_DRAWINGS_CURRENT END)
        / NULLIF(AVG(AMT_DRAWINGS_CURRENT), 0) AS cc_drawings_change_ratio,
    AVG(CASE WHEN MONTHS_BALANCE >= -12 THEN paid_min_only END) AS cc_pct_min_payment_12m,
    AVG(CASE WHEN MONTHS_BALANCE >= -12 THEN SK_DPD > 0 END) AS cc_pct_dpd_12m,
    AVG(CASE WHEN MONTHS_BALANCE >= -12 THEN SK_DPD > 0 END) - AVG(SK_DPD > 0) AS cc_dpd_trend_12m
FROM c
GROUP BY SK_ID_CURR
"""

cc_change = run_query("Credit Card Change", CC_CHANGE_SQL)
cc_change.describe().T

Credit Card Change: 103,558 rows x 11 columns in 9s


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,103558.0,278381.457956,102779.519683,100006.000000,189536.250000,278649.000000,367690.000000,456250.000000
cc_n_months_12m,103558.0,10.311671,2.558131,1.000000,10.000000,11.000000,12.000000,36.000000
cc_util_3m,77751.0,0.347040,0.428874,0.000000,0.000000,0.000000,0.854494,8.100310
cc_util_12m,84881.0,0.294425,0.369034,0.000000,0.000000,0.010389,0.631812,2.708565
cc_util_trend_3m,77751.0,0.008669,0.244445,-1.087329,-0.035843,0.000000,0.110901,6.843516
cc_pct_maxed_12m,84881.0,0.190326,0.316275,0.000000,0.000000,0.000000,0.333333,1.000000
cc_balance_change_3m,98576.0,11110.461202,71757.674448,-916486.507500,0.000000,0.000000,2929.982500,844769.103000
cc_drawings_change_ratio,69146.0,0.448652,1.178702,0.000000,0.000000,0.000000,0.441311,47.500000
cc_pct_min_payment_12m,44255.0,0.265454,0.316153,0.000000,0.000000,0.142857,0.500000,1.000000
cc_pct_dpd_12m,103558.0,0.012250,0.090963,0.000000,0.000000,0.000000,0.000000,1.000000


# Curent loans vs previous borrowing, and recent credit seeking

The purpose of this data frame is going to look at two things within applicants:

1. Whether the loan being requested now is ussually large compared with what the applicant has previously borrowed
2. Whether they've been applying for and taking out credit more often....

In [ ]:
BO